# Module 01 - Scalar Autodiff Playground

Use this notebook after your `g2c.autodiff.Value` tests are passing. It gives you a place to do the Module 01 written/code exercises interactively without pasting large code blocks into your answer notes.

Goal: use only the `Value` class you built. No PyTorch, no NumPy for the model logic.

1. Read the lesson page (`docs/modules/01-autodiff.md`).
2. Open this notebook with `.venv/bin/python scripts/open_notebook.py 01`.
3. Answer the `Question:` / `Answer:` cells below.
4. When you're ready, ask a coding agent to grade your notebook.

Partial work is fine. Blank `Answer: ""` strings are skipped, not counted wrong. If you'd like a hint instead of a grade, write the request inline in the answer string and the agent will tutor first.

## Setup

If you edit `g2c/autodiff/*.py` while this notebook is open, restart the kernel or use autoreload.

In [ ]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations

import random

import matplotlib.pyplot as plt

from g2c.autodiff import (
    ScalarMLP,
    Value,
    numerical_grad,
    single_neuron_forward,
    train_xor_step,
    xor_loss,
)
from g2c.notebook_extras.autodiff import plot_xor_decision_snapshots

random.seed(42)


In [ ]:
# Quick sanity check: this should run without raising.
a = Value(2.0)
b = Value(3.0)
c = a * b + a.tanh()
c.backward()
print(c)
print("dc/da =", a.grad)
print("dc/db =", b.grad)

## Exercise 1 — Forward And Backward By Hand

Expression:

`f = (a * b + b**2) * tanh(c)` with `a = 1`, `b = 2`, `c = 0.5`.

First compute the forward value and gradients by hand. Then put your expected values below and compare them against your engine.

In [ ]:
# TODO: replace None with your hand-computed values.
expected_f = None
expected_da = None
expected_db = None
expected_dc = None

In [ ]:
a = Value(1.0)
b = Value(2.0)
c = Value(0.5)

f = (a * b + b**2) * c.tanh()
f.backward()

print("f.data =", f.data, "expected =", expected_f)
print("a.grad =", a.grad, "expected =", expected_da)
print("b.grad =", b.grad, "expected =", expected_db)
print("c.grad =", c.grad, "expected =", expected_dc)

## Exercise 2 — Gradient Checking

Use your `numerical_grad` helper to compare finite differences against `.backward()` on a few expressions.

In [ ]:
def analytic_grad(f, x_data: float) -> float:
    """Return df/dx from your autodiff engine for a scalar function f(Value) -> Value."""
    x = Value(x_data)
    y = f(x)
    y.backward()
    return x.grad


checks = [
    ("quadratic", lambda x: x * x + 3 * x + 1, 2.0),
    # TODO: add at least three more nontrivial expressions.
    # ("tanh_mix", lambda x: ..., 0.7),
    # ("exp_log_mix", lambda x: ..., 1.5),
    # ("relu_pow_mix", lambda x: ..., -0.5),
]

for name, f, x_data in checks:
    x = Value(x_data)
    numeric = numerical_grad(f, x)
    analytic = analytic_grad(f, x_data)
    print(f"{name:>12}: analytic={analytic:.8f} numeric={numeric:.8f} diff={abs(analytic - numeric):.2e}")

## Exercise 3 — A Single Neuron From Scratch

Build `y = tanh(w1*x1 + w2*x2 + b)`, compute squared-error loss, and manually update `w1`, `w2`, and `b` once.

In [ ]:
w1 = Value(0.2)
w2 = Value(-0.3)
b = Value(0.1)
target = 1.0


def train_single_neuron_step():
    y = single_neuron_forward(0.75, 0.25, w1, w2, b)
    loss = (y - target) ** 2
    loss.backward()

    print("y =", y.data)
    print("loss =", loss.data)
    print("grads:", w1.grad, w2.grad, b.grad)
    print()


train_single_neuron_step()

lr = 0.1
for param in (w1, w2, b):
    param.data -= lr * param.grad
    param.grad = 0.0

print("after one manual gradient step")
train_single_neuron_step()


## Exercise 4 — XOR With A Tiny MLP

Build a 2-2-1 MLP using only `Value`:

- 2 inputs
- 2 hidden tanh neurons
- 1 tanh output neuron

The target labels below use `-1` and `1`, which match tanh's output range better than `0` and `1`.

In [ ]:
X = [
    (0.0, 0.0),
    (0.0, 1.0),
    (1.0, 0.0),
    (1.0, 1.0),
]
Y = [-1.0, 1.0, 1.0, -1.0]

list(zip(X, Y))

In [ ]:
rng = random.Random(42)
demo_model = ScalarMLP(rng=rng)

print("ScalarMLP lives in g2c/autodiff/nn.py so your work stays in the editor.")
print("parameter count:", len(demo_model.parameters()))
print("initial predictions:")
for x, y_true in zip(X, Y):
    print(f"x={x} target={y_true:>4} pred={demo_model(x).data: .3f}")


In [ ]:
demo_loss = xor_loss(demo_model)
print("initial XOR loss:", demo_loss.data)
print("one training step returned loss:", train_xor_step(demo_model, lr=0.1))
print("loss after one update:", xor_loss(demo_model).data)


In [ ]:
rng = random.Random(42)
model = ScalarMLP(rng=rng)

losses = []
for step in range(500):
    loss_value = train_xor_step(model, lr=0.1)
    losses.append(loss_value)

print("initial loss:", losses[0])
print("final loss:", losses[-1])

for x, y_true in zip(X, Y):
    pred = model(x).data
    print(f"x={x} target={y_true:>4} pred={pred: .3f}")


In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(losses)
plt.xlabel("step")
plt.ylabel("XOR loss")
plt.title("Module 01 XOR training")
plt.grid(True, alpha=0.3);

### Visualize the XOR boundary

The model output is a continuous surface over `(x1, x2)`. The black line is the decision boundary where the tanh output crosses 0; the four markers are the XOR training points.


In [ ]:
plot_xor_decision_snapshots(steps=(0, 100, 300, 500), seed=42, lr=0.1)


## Exercise 5 — Topology Stress Test

Build expressions that reuse the same `Value` multiple times. This should catch backward implementations that overwrite gradients instead of accumulating them.

In [ ]:
stress_cases = [
    ("a*a + a", lambda a: a * a + a, lambda x: 2 * x + 1, 3.0),
    # TODO: add at least three more shared-node expression.
]

for name, f, expected_grad, x_data in stress_cases:
    a = Value(x_data)
    out = f(a)
    out.backward()
    print(f"{name:>12}: grad={a.grad:.6f} expected={expected_grad(x_data):.6f}")

## Submission Notes

When complete launch a coding agent like Codex or Claude and ask it to grade your exercise answers for Module 01. You can also use an agent to ask questions or grade partial answers.